# common coordinate axis synthetic dataset validation

Set STP_PROJECT before launching Jupyter. The bundle README lists inputs, kernels and run order. Run the sections for your workflow.

In [ ]:
import os
from pathlib import Path

# Set these before starting Jupyter. Existing per-notebook overrides still apply.
if "STP_PROJECT" not in os.environ:
    raise RuntimeError("Set STP_PROJECT to the experiment workspace before starting Jupyter.")
project_root = Path(os.environ["STP_PROJECT"]).expanduser().resolve()
os.environ["STP_PROJECT"] = str(project_root)
os.environ.setdefault("SPATIAL_TI_PROJECT", str(project_root))
os.environ.setdefault("PFC_PROJECT", str(project_root))
os.environ.setdefault("FLOW_STP_REPO", str(project_root))
os.environ.setdefault("QIAN_PFC_DATA_ROOT", str(project_root / "data/qian_pfc"))
os.environ.setdefault("QIAN_DATA_ROOT", os.environ["QIAN_PFC_DATA_ROOT"])
os.environ.setdefault("PFC_DATA_ROOT", os.environ["QIAN_PFC_DATA_ROOT"])
os.environ.setdefault("STVCR_REPO", str(project_root / "external/stVCR"))

## Score anatomical ordering and export Figure 4A inputs

In [ ]:
# Publication figure: synthetic anatomical-coordinate recovery, dataset seed 20261023.
# Reads the newest FROZEN Flow-STP Stage-1 results and evaluator-only ground truth.
# No model fitting, retraining, hyperparameter tuning, or per-tissue orientation fitting.
# Absolute Spearman uses RAW coordinates, unchanged by any display scaling.
# ONE GLOBAL sign aligns orientation across all tissues. Then EACH tissue's
# ground-truth and globally oriented learned z are independently min-max mapped
# to [0, 1] FOR COLOR DISPLAY ONLY. No per-tissue sign flips, no change to scores.

from pathlib import Path
import hashlib
import json
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

PROJECT = Path(os.environ.get(
    "STP_PROJECT", os.environ["STP_PROJECT"]
)).expanduser()
DATA = Path(os.environ.get(
    "CRESCENT_DATA_ROOT", str(PROJECT / "Experiments/simulation validation/"
                                    "Synthetic benchmarking/experiments/"
                                    "synthetic_developmental_crescent_v1")
)).expanduser()
RUN = Path(os.environ.get(
    "FLOW_STP_ALL_TEN_OUT", str(PROJECT / "experiments/synthetic_benchmark/"
                                     "synthetic_developmental_crescent_v1_flow_stp/"
                                     "all_ten_tissues_stage1_stage2_v1")
)).expanduser()
FIG_DIR = Path(os.environ.get(
    "SYNTHETIC_Z_FIG_DIR", str(PROJECT / "experiments/synthetic_benchmark/"
                                   "publication_results/"
                                   "synthetic_anatomical_z_seed20261023_paper_compact")
)).expanduser()
SHOW_TISSUE = 7
PINNED_COMMIT = "76bb952a841c3671ec35161149e780167fd9b7b1"
EXPECTED_COUNTS = [1800, 2093, 2419, 2784, 3193, 3653, 4171, 4755, 5411, 6146]


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def require(path):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"Required frozen data/model output not found: {path}")
    return path


# (1) Validate fixed training identity and data identity BEFORE using hidden truth.
manifest_path = require(DATA / "manifests/dataset_manifest.json")
manifest = json.loads(manifest_path.read_text())
assert manifest["dataset_id"] == "synthetic_developmental_crescent_v1", "Wrong synthetic dataset"
assert int(manifest["seed"]) == 20261023, "Wrong dataset seed; refusing stale results"
assert manifest["status"] == "frozen", "Dataset not frozen"
assert int(manifest["n_observed_cells"]) == 36425

run_config_path = require(RUN / "run_config.json")
run_config = json.loads(run_config_path.read_text())
assert run_config["dataset_id"] == "synthetic_developmental_crescent_v1"
assert run_config.get("uses_hidden_truth") is False, "Truth was used for model training"
assert run_config["source"]["commit"] == PINNED_COMMIT, "Not the newest pinned Flow-STP fit"

done_path = require(RUN / "complete.json")
done = json.loads(done_path.read_text())
assert done["completed"] is True
assert done["source_commit"] == PINNED_COMMIT
assert int(done["n_tissues"]) == 10 and int(done["n_cells"]) == 36425

public_rel = "train_only/public/observed_cells.csv.gz"
truth_rel = "evaluator_only/observed_hidden/cell_truth.csv.gz"
observed_path = require(DATA / public_rel)
truth_path = require(DATA / truth_rel)
z_rel = "stage1_native/tables/obs_with_stage1a.csv.gz"
z_path = require(RUN / z_rel)

for path, expected in [
    (observed_path, manifest["file_sha256"][public_rel]),
    (truth_path, manifest["file_sha256"][truth_rel]),
    (z_path, done["file_sha256"][z_rel]),
]:
    measured = sha256(path)
    assert measured == expected, f"SHA-256 mismatch, refusing modified input: {path}"

# (2) Align by *cell_id*, NEVER assume a model output and truth happen to share row order.
obs = pd.read_csv(observed_path, usecols=["cell_id", "tissue_id", "time", "x", "y"])
truth = pd.read_csv(truth_path, usecols=["cell_id", "z_true"])
learned = pd.read_csv(z_path, usecols=["cell_id", "z_stage1a"])
for frame in (obs, truth, learned):
    frame["cell_id"] = frame["cell_id"].astype(str)
    assert frame.cell_id.is_unique, "Duplicate cell ID: ambiguous truth or prediction matching"

assert len(obs) == sum(EXPECTED_COUNTS)
assert set(obs.cell_id) == set(truth.cell_id) == set(learned.cell_id), (
    "Observed, hidden-truth, and trained-Stage-1 cell-ID sets differ")
joined = obs.merge(truth, on="cell_id", how="left", validate="one_to_one", sort=False)
joined = joined.merge(learned, on="cell_id", how="left", validate="one_to_one", sort=False)
assert len(joined) == 36425
assert joined[["x", "y", "z_true", "z_stage1a"]].notna().all().all()
assert np.isfinite(joined[["x", "y", "z_true", "z_stage1a"]].to_numpy(float)).all()
assert np.array_equal(np.sort(joined.tissue_id.unique()), np.arange(10))
assert joined.groupby("tissue_id").size().reindex(range(10)).tolist() == EXPECTED_COUNTS
np.testing.assert_allclose(joined["time"], joined["tissue_id"] / 9., atol=1e-10, rtol=0)

# (3) Evaluate *raw* predictions; determine one GLOBAL sign solely for presentation.
# Ground truth is used here for scoring/figure orientation, never model fitting.
pooled_rho = float(spearmanr(joined.z_stage1a, joined.z_true).statistic)
assert np.isfinite(pooled_rho)
global_sign = -1 if pooled_rho < 0 else +1
z_oriented = global_sign * joined.z_stage1a.to_numpy(float)
joined["z_oriented_for_display"] = z_oriented

# Scale each tissue's own z_true and globally oriented learned z INDEPENDENTLY.
# This makes both colorbars span 0..1 in the displayed tissue even when learned
# z occupies only part of the global range. No z normalization enters metrics.
joined["z_true_display"] = np.nan
joined["learned_z_display"] = np.nan

rows = []
color_scaling_bounds = {}
for tissue_id in range(10):
    mask = joined.tissue_id.eq(tissue_id)
    part = joined.loc[mask]
    rho = float(spearmanr(part.z_stage1a, part.z_true).statistic)
    assert np.isfinite(rho), f"Undefined Spearman in tissue {tissue_id}"
    truth_vals = part.z_true.to_numpy(dtype=float)
    learned_vals = part.z_oriented_for_display.to_numpy(dtype=float)
    truth_lo, truth_hi = float(truth_vals.min()), float(truth_vals.max())
    learned_lo, learned_hi = float(learned_vals.min()), float(learned_vals.max())
    assert truth_hi > truth_lo, f"Collapsed ground-truth z in tissue {tissue_id}"
    assert learned_hi > learned_lo, f"Collapsed learned z in tissue {tissue_id}"
    joined.loc[mask, "z_true_display"] = (truth_vals - truth_lo) / (truth_hi - truth_lo)
    joined.loc[mask, "learned_z_display"] = (learned_vals - learned_lo) / (learned_hi - learned_lo)
    color_scaling_bounds[str(tissue_id)] = {
        "z_true_min_max": [truth_lo, truth_hi],
        "globally_oriented_learned_z_min_max": [learned_lo, learned_hi],
    }
    rows.append({"tissue_id": tissue_id, "n_cells": len(part),
                 "spearman_raw": rho, "abs_spearman": abs(rho),
                 "spearman_after_global_orientation": global_sign * rho,
                 "global_sign_for_display": global_sign,
                 "truth_display_min": truth_lo, "truth_display_max": truth_hi,
                 "learned_display_min": learned_lo, "learned_display_max": learned_hi})
metrics = pd.DataFrame(rows)
assert joined[["z_true_display", "learned_z_display"]].notna().all().all()
assert joined[["z_true_display", "learned_z_display"]].to_numpy().min() >= -1e-12
assert joined[["z_true_display", "learned_z_display"]].to_numpy().max() <= 1 + 1e-12
misoriented = metrics.loc[metrics.spearman_after_global_orientation < 0, "tissue_id"].tolist()
if misoriented:
    warnings.warn("The following tissues have opposite orientation despite the ONE global "
                  f"sign flip: {misoriented}. Absolute correlation bars hide this distinction; "
                  "inspect the signed metrics CSV. No per-tissue flips were performed.")

# (4) SLOPER-inspired, compact publication panels (one full-width and one one-column).
# Display color values use within-tissue min-max scaling; the correlations use RAW z.
# Fonts are specified in physical points at final manuscript width (7.05in or 3.35in).
# Labels/titles are kept short: the biological interpretation belongs in the caption.
sel = joined.loc[joined.tissue_id.eq(SHOW_TISSUE)]
val = float(metrics.loc[metrics.tissue_id.eq(SHOW_TISSUE), "abs_spearman"].iloc[0])

FIG_DIR.mkdir(parents=True, exist_ok=True)

# Consistent paper typography. At printed size, 8-10 pt matches/exceeds body text.
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 9.0,
    "axes.titlesize": 10.0,
    "axes.labelsize": 9.5,
    "xtick.labelsize": 8.0,
    "ytick.labelsize": 8.0,
    "pdf.fonttype": 42,   # editable text in PDF/Illustrator/PowerPoint
    "ps.fonttype": 42,
    "svg.fonttype": "none",
    "savefig.facecolor": "white",
})
BLUE = "#4C78A8"
EXPORT_DPI = 400


def export_figure(fig, path_no_extension):
    for suffix, kwargs in [(".pdf", {}), (".svg", {}),
                           (".png", {"dpi": EXPORT_DPI})]:
        dest = path_no_extension.with_suffix(suffix)
        fig.savefig(dest, facecolor="white", **kwargs)
        print("SAVED", dest)


def plot_tissue(ax, z_values, title):
    # Same actual XY locations in both tissue panels.
    scatter = ax.scatter(sel.x.to_numpy(), sel.y.to_numpy(),
                         c=z_values, cmap="viridis", vmin=0, vmax=1,
                         s=1.9, linewidths=0, rasterized=True)
    ax.set_aspect("equal", adjustable="box")
    ax.set_axis_off()
    ax.set_title(title, fontsize=10.0, fontweight="bold", pad=3.5)
    ax.margins(x=.02, y=.02)
    return scatter


# MAIN FIGURE: full two-column width (7.05 inches), only 2.75 inches high.
# One horizontal row of A: truth, B: learned, C: per-tissue accuracy.
fig = plt.figure(figsize=(7.05, 2.75), facecolor="white")
fig.text(.016, .97, "Synthetic dataset", fontsize=11.0, fontweight="bold",
         ha="left", va="top")
a_true = fig.add_axes([.014, .25, .278, .59])
a_learn = fig.add_axes([.338, .25, .278, .59])
a_bar = fig.add_axes([.714, .265, .277, .54])

h1 = plot_tissue(a_true, sel.z_true_display.to_numpy(), r"A   Ground-truth $z$")
h2 = plot_tissue(a_learn, sel.learned_z_display.to_numpy(), r"B   Learned $z$")

# Very slim colorbars with just 0 and 1: both coordinates are separately
# normalized within the displayed tissue for COLOR ONLY.
c1 = fig.add_axes([.297, .365, .010, .36])
c2 = fig.add_axes([.623, .365, .010, .36])
for artist, cax in [(h1, c1), (h2, c2)]:
    cb = fig.colorbar(artist, cax=cax, ticks=[0, .5, 1])
    cb.ax.set_yticklabels(["0", ".5", "1"])
    cb.ax.tick_params(labelsize=7, width=.6, length=2, pad=1)
    cb.outline.set_linewidth(.5)

# Small, useful panel-specific metric: not a held-out cell evaluation.
a_learn.text(.5, -.09, rf"Tissue {SHOW_TISSUE}   $|\rho|={val:.3f}$",
             ha="center", va="top", transform=a_learn.transAxes,
             fontsize=8.7)

# SLOPER-like close bars and rotated BOLD category labels.
xx = np.arange(len(metrics), dtype=float)
y = metrics.sort_values("tissue_id").abs_spearman.to_numpy(float)
a_bar.bar(xx, y, width=.90, color=BLUE, edgecolor="none", zorder=2)
a_bar.set_xlim(-.48, len(xx)-.52)
a_bar.set_ylim(0, 1.025)
a_bar.set_yticks([0, .25, .5, .75, 1.0])
a_bar.set_yticklabels(["0", ".25", ".5", ".75", "1"])
a_bar.set_xticks(xx)
a_bar.set_xticklabels([f"Tissue {int(i)}" for i in xx],
                      rotation=69, ha="right", va="top",
                      fontsize=8.1, fontweight="bold", rotation_mode="anchor")
a_bar.set_ylabel(r"$|\rho|$ with ground-truth $z$ axis",
                 fontsize=9.4, labelpad=3)
a_bar.set_title(r"C   $|\rho|$ by tissue", fontsize=10,
                fontweight="bold", pad=5)
a_bar.tick_params(axis="x", pad=1, length=0)
a_bar.tick_params(axis="y", pad=1.5, width=.7, length=2.5)
a_bar.spines[["top", "right"]].set_visible(False)
a_bar.spines[["left", "bottom"]].set_linewidth(.65)
a_bar.grid(axis="y", linewidth=.45, alpha=.28, zorder=0)
a_bar.set_axisbelow(True)

out_main = FIG_DIR / "synthetic_seed20261023_z_paper_compact_3panel"
export_figure(fig, out_main)
plt.show()
plt.close(fig)

# ALSO EXPORT a standalone single-column bar panel for flexible manuscript layouts.
# The same metrics, same axis and labels, same exact blue hue.
fig2 = plt.figure(figsize=(3.35, 2.75), facecolor="white")
ax = fig2.add_axes([.24, .30, .745, .59])
ax.bar(xx, y, width=.90, color=BLUE, edgecolor="none")
ax.set_xlim(-.48, len(xx)-.52)
ax.set_ylim(0, 1.025)
ax.set_yticks([0, .25, .5, .75, 1])
ax.set_yticklabels(["0", ".25", ".5", ".75", "1"])
ax.set_xticks(xx)
ax.set_xticklabels([f"Tissue {int(i)}" for i in xx],
                   rotation=69, ha="right", va="top", fontweight="bold",
                   fontsize=8.3, rotation_mode="anchor")
ax.set_ylabel(r"$|\rho|$ with ground-truth $z$ axis", fontsize=9.5)
ax.set_title("Anatomical-order recovery", fontsize=10, fontweight="bold", pad=5)
ax.tick_params(axis="x", length=0, pad=2)
ax.tick_params(axis="y", length=2.5, pad=1.5)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=.28, linewidth=.45)
ax.set_axisbelow(True)
out_bar = FIG_DIR / "synthetic_seed20261023_z_paper_bar_only"
export_figure(fig2, out_bar)
plt.show()
plt.close(fig2)

# (5) Reproducible numeric and audit outputs; do not alter frozen training artifacts.
metrics_file = FIG_DIR / "per_tissue_absolute_spearman.csv"
metrics.to_csv(metrics_file, index=False, float_format="%.12g")
sel[["cell_id", "tissue_id", "x", "y", "z_true", "z_stage1a",
     "z_oriented_for_display", "z_true_display", "learned_z_display"]].to_csv(
    FIG_DIR / "tissue_07_figure_data.csv.gz", index=False, compression="gzip")
provenance = {
    "dataset_id": manifest["dataset_id"], "dataset_seed": manifest["seed"],
    "source_commit": PINNED_COMMIT, "n_tissues": 10, "n_cells": len(joined),
    "input_files_sha256": {str(path): sha256(path) for path in
        [manifest_path, observed_path, truth_path, run_config_path, done_path, z_path]},
    "orientation_rule": "One global sign from pooled raw Spearman; NO per-tissue sign flips",
    "global_sign_for_display": global_sign, "pooled_raw_spearman": pooled_rho,
    "display_scaling_rule": "Within EACH tissue, independently min-max scale true z and globally oriented learned z to [0,1]; display ONLY, never scores",
    "display_scale_bounds_by_tissue": color_scaling_bounds,
    "tissues_with_opposite_signed_orientation_after_global_flip": misoriented,
    "score_definition": "abs(Spearman(raw learned z_stage1a, hidden z_true)) per observed tissue",
    "evaluated_cells": "all 36425 observed cells; hidden z_true was never fitted",
    "not_a_cell_holdout": True, "rmse_reported": False,
    "figure_tissue": SHOW_TISSUE, "hidden_truth_used_only_for_evaluation": True,
    "figure_layout": "7.05 x 2.75 inches three horizontal panels; optional 3.35 x 2.75 inch bar-only",
    "bar_style": "Blue, width 0.90, bold rotated tissue labels, y axis |rho| with ground-truth z axis",
    "training_artifacts_modified": False,
}
(FIG_DIR / "figure_provenance.json").write_text(
    json.dumps(provenance, indent=2, sort_keys=True) + "\n")
print("\n=== SYNTHETIC DATASET: ABSOLUTE SPEARMAN BY TISSUE ===")
print(metrics.to_string(index=False, float_format=lambda x: f"{x:.6f}"))
print(f"Pooled raw Spearman: {pooled_rho:+.6f} | ONE GLOBAL display sign: {global_sign:+d}")
print(f"Mean of 10 tissue absolute Spearman values: {metrics.abs_spearman.mean():.6f}")
print(f"Opposite-orientation tissues after global sign: {misoriented}")
print("Display: independent per-tissue min-max [0,1] on truth and globally oriented learned z; metrics UNCHANGED")
print(f"\nFIGURE AND METRICS: {FIG_DIR}")